
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Demo: HPO with Ray Tune

In this demo, you will learn how to use **Ray Tune** — a powerful hyperparameter optimization framework — to tune machine learning models in **Databricks**. 

We will demonstrate how to implement the Ray Tune framework using a **Random Forest Regressor** from Scikit-Learn, covering:
- **Defining search spaces**
- **Creating objective functions**
- **Optimizing hyperparameters** 


Additionally, we will track and log the results using **MLflow**, enabling efficient management and monitoring of the tuning process.

---

## **Learning Objectives**

By the end of this demo, you will be able to:
- Define an **objective function** specific to Ray Tune.
- Set up **Optuna-style search spaces** within Ray Tune.
- Configure **Ray Tune's Tuner and compute resources**.
- Optimize hyperparameters using **parallel execution**.


# Model Tuning with Ray Tune and a Single-Machine Model
In this part, we will use **Ray for distributed hyperparameter optimization** while training a **Scikit-Learn** model.

### How This Works:
- **Data is converted from a Spark DataFrame to Pandas** to enable single-machine model training.
- **Ray Tune runs on a single machine** but distributes trial execution across *multiple* CPU threads to speed up hyperparameter tuning.
- **MLflow tracks the experiment**, logging the best hyperparameters and model performance.

This approach allows us to use **Ray for distributed hyperparameter search**, while **training the model on a single node** to take advantage of Scikit-Learn’s efficient implementations.


## REQUIRED - SELECT CLASSIC COMPUTE
Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:
1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.
   
   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## Requirements

Please review the following requirements before starting the lesson:

* To run this notebook, you need a classic cluster running one of the following Databricks runtime(s): **16.3.x-cpu-ml-scala2.12**. **Do NOT use serverless compute to run this notebook**.

In [0]:
%pip install -U optuna optuna-integration mlflow
%pip install --upgrade ray[tune]
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Before starting the demo, run the provided classroom setup script.

In [0]:
%run ../Includes/Classroom-Setup-02.1b

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
print(f" Training coount:{train_df.count()}")
print(f" Testing coount:{test_df.count()}")
print(f" Feature Columns:{feature_columns}")
print(f" Label Column : {label_column}")


train_df.toPandas().head(10)
# train_df.toPandas()[feature_columns].head(10)
# train_df.toPandas()[label_column].head(10)

 Training coount:929
 Testing coount:215
 Feature Columns:['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH', 'sulphates', 'alcohol']
 Label Column : quality


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,features,ID
0,5.0,0.380,0.01,1.60,0.048,26.0,60.0,0.99084,3.70,0.75,14.000000,6,"[5.0, 0.38, 0.01, 1.6, 0.048, 26.0, 60.0, 0.99...",0
1,5.0,0.400,0.50,4.30,0.046,29.0,80.0,0.99020,3.49,0.66,13.600000,6,"[5.0, 0.4, 0.5, 4.3, 0.046, 29.0, 80.0, 0.9902...",1
2,5.1,0.420,0.00,1.80,0.044,18.0,88.0,0.99157,3.68,0.73,13.600000,7,"[5.1, 0.42, 0.0, 1.8, 0.044, 18.0, 88.0, 0.991...",3
3,5.2,0.645,0.00,2.15,0.080,15.0,28.0,0.99444,3.78,0.61,12.500000,6,"[5.2, 0.645, 0.0, 2.15, 0.08, 15.0, 28.0, 0.99...",4
4,5.3,0.470,0.11,2.20,0.048,16.0,89.0,0.99182,3.54,0.88,13.566667,7,"[5.3, 0.47, 0.11, 2.2, 0.048, 16.0, 89.0, 0.99...",5
5,5.3,0.715,0.19,1.50,0.161,7.0,62.0,0.99395,3.62,0.61,11.000000,5,"[5.3, 0.715, 0.19, 1.5, 0.161, 7.0, 62.0, 0.99...",7
6,5.4,0.580,0.08,1.90,0.059,20.0,31.0,0.99484,3.50,0.64,10.200000,6,"[5.4, 0.58, 0.08, 1.9, 0.059, 20.0, 31.0, 0.99...",9
7,5.4,0.740,0.09,1.70,0.089,16.0,26.0,0.99402,3.67,0.56,11.600000,6,"[5.4, 0.74, 0.09, 1.7, 0.089, 16.0, 26.0, 0.99...",10
8,5.6,0.310,0.78,13.90,0.074,23.0,92.0,0.99677,3.39,0.48,10.500000,6,"[5.6, 0.31, 0.78, 13.9, 0.074, 23.0, 92.0, 0.9...",11
9,5.6,0.540,0.04,1.70,0.049,5.0,13.0,0.99420,3.72,0.58,11.400000,5,"[5.6, 0.54, 0.04, 1.7, 0.049, 5.0, 13.0, 0.994...",12


Successfully saved table: dbacademy.labuser11731658_1758653716.wine_quality_features


**Other Conventions:**

Throughout this demo, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets.wine_quality}")

Username:          labuser11731658_1758653716@vocareum.com
Catalog Name:      dbacademy
Schema Name:       labuser11731658_1758653716
Working Directory: /Volumes/dbacademy/ops/labuser11731658_1758653716@vocareum_com
Dataset Location:  /Volumes/dbacademy_wine_quality/v01


## Configure a Ray Cluster
Let's begin by setting up a single-machine (driver-only) Ray cluster by defining the following:
- 3 CPU cores allocated for the head node, leaving 1 core for Spark
> The Vocareum environment allocates 4 CPUs per user for this demonstration. 
- 1 worker node
- 4 CPUs per worker

Additionally, we will initialize the Ray cluster on spark with the above configurations with `ray.init()` and defining the environment variable `RAY_ADDRESS`.

In [0]:
import os
import ray
from ray.util.spark import setup_ray_cluster, shutdown_ray_cluster

# Attempt to shut down any existing Ray cluster
try:
    shutdown_ray_cluster()
    print("Existing Ray cluster shutdown successfully.")
except Exception as e:
    print(f"Warning: No active Ray cluster to shut down. Details: {e}")

# Set up configurations for a single-machine (driver-only) Ray cluster
num_cpus_head_node = 3  # Use 3 CPU cores, leaving 1 for Spark
num_worker_nodes = 1  # Single-node setup (driver only)
num_cpu_cores_per_worker = 4  # Unused since there's only one worker

# Initialize the Ray cluster on Spark
ray_conf = setup_ray_cluster(
    min_worker_nodes=num_worker_nodes,  
    max_worker_nodes=num_worker_nodes,
    num_cpus_head_node=num_cpus_head_node,  
    num_gpus_head_node=0  # No GPU usage
)

# Initialize Ray with the configured settings
ray.init(ignore_reinit_error=True)
print(f"Ray initialized with address: {ray_conf[0]}")

# Set Ray address for Spark integration
os.environ['RAY_ADDRESS'] = ray_conf[0]

In each spark worker node, we recommend making the sum of 'spark_executor_memory + num_Ray_worker_nodes_per_spark_worker * (memory_worker_node + object_store_memory_worker_node)' to be less than 'spark_worker_physical_memory * 0.8', otherwise it might lead to spark worker physical memory exhaustion and Ray task OOM errors.

2025-09-23 19:42:58,797	WARNING cluster_init.py:1131 -- The provided CPU resources for each ray worker are inadequate to start a ray cluster. Based on the total cpu resources available and the configured task sizing, each ray worker node would start with 1 CPU cores. This is less than the recommended value of `4` CPUs per worker. On spark version >= 3.4 or Databricks Runtime 12.x, you can set the argument `num_cpus_worker_node` to a value >= 4 to address it, otherwise you need to increase the spark application configuration 'spark.task.cpus' to a minimum of `4` to address it.
The provided memory resources for each ray worker node are inadequate. Based on the total memory available on the spark cluster and the configured task sizing, each ray worker would start with 3742829118 bytes heap memory. This is less than the recommended value of 10GB. The ray worker node heap memory size is calculated by (SPARK_WORKER_PHYSICAL_MEMORY / num_local_spark_task_slots * 0.8) - object_store_memory_wor

MLflow support is not correctly configured within Ray tasks.To enable MLflow integration, you need to set environmental variables DATABRICKS_HOST + DATABRICKS_TOKEN, or set environmental variables DATABRICKS_HOST + DATABRICKS_CLIENT_ID + DATABRICKS_CLIENT_SECRET before calling `ray.util.spark.setup_ray_cluster`, these variables are used to set up authentication with Databricks MLflow service. For details, you can refer to Databricks documentation at Databricks PAT auth or Databricks OAuth .

2025-09-23 19:43:01,328	WARNING node.py:1806 -- The object spilling config is specified from an unstable API - system config or environment variable. This is subject to change in the future. You can use the stable API - --object-spilling-directory in ray start or object_spilling_directory in ray.init() to specify the object spilling directory instead. If you need more advanced settings, please open a github issue with the Ray team.
2025-09-23 19:43:01,327	INFO usage_lib.py:473 -- Usage stats collection is enabled by default without user confirmation because this terminal is detected to be non-interactive. To disable this, add `--disable-usage-stats` to the command that starts the cluster, or run the following command: `ray disable-usage-stats` before starting the cluster. See https://docs.ray.io/en/master/cluster/usage-stats.html for more details.
2025-09-23 19:43:01,327	INFO scripts.py:913 -- Local node IP: 10.3.33.157
2025-09-23 19:43:03,168	SUCC scripts.py:949 -- -------------------

2025-09-23 19:43:18,829	INFO cluster_init.py:693 -- Ray head node started.
2025-09-23 19:43:18,831	INFO databricks_hook.py:142 -- The Ray cluster will be shut down automatically if you don't run commands on the Databricks notebook for 30.0 minutes. You can change the auto-shutdown minutes by setting 'DATABRICKS_RAY_ON_SPARK_AUTOSHUTDOWN_MINUTES' environment variable, setting it to 0 means that the Ray cluster keeps running until you manually call `ray.util.spark.shutdown_ray_cluster()` or detach Databricks notebook.
2025-09-23 19:43:18,840	INFO worker.py:1771 -- Connecting to existing Ray cluster at address: 10.3.33.157:9813...
2025-09-23 19:43:18,855	INFO worker.py:1951 -- Connected to Ray cluster.
2025-09-23 19:44:21,684	WARNING cluster_init.py:143 -- Dependencies to launch the optional dashboard API server cannot be found. They can be installed with pip install ray[default], root cause: (ModuleNotFoundError("No module named 'opentelemetry.exporter'"))
2025-09-23 19:44:24,724	INFO cl

Ray initialized with address: 10.3.33.157:9813


### Data Preparation for Distributed Optuna with Single-Machine Training

In this task, before we define the objective function, we need to convert our Spark DataFrame to a Pandas DataFrame. This will allow us to split the data into training and test sets and standardize the features. These steps are essential for conducting single-node model training using **Scikit-learn**.

**Instructions**:

1. **Convert the Spark DataFrame** into a Pandas DataFrame for single-node processing.
2. **Split the dataset** into training and test sets using **Scikit-learn**'s `train_test_split` method.
3. **Standardize the features** to ensure the model performs optimally.


In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Convert Spark DataFrame to Pandas for single-machine training
train_pandas = train_df.toPandas()

# Separate features and labels
X = train_pandas[feature_columns]
y = train_pandas[label_column]

# Split the data into training and test sets using Scikit-learn
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Display the shapes of the training and test sets
print(f"Training data shape: {X_train.shape}, Test data shape: {X_test.shape}")

Training data shape: (743, 11), Test data shape: (186, 11)


### Define the Ray Tune Objective Function and Search Space for Distributed Hyperparameter Search (Single-Machine Training)

In this step, we will define the **objective function** for hyperparameter tuning using **Ray Tune**. 

This function will:
- Suggest hyperparameters dynamically.
- Train and evaluate a `RandomForest` model using **Scikit-learn**.
- Utilize **cross-validation** for performance evaluation.

#### **Key Differences from the Previous Demonstration**
- Unlike the previous approach, we now integrate **cross-validation** for model evaluation.
- Cross-validation is feasible in **single-node** training but can be **resource-intensive** in a distributed setup.

---

### **Instructions**
1. **Define the Hyperparameter Search Space**  
   - Use **Ray Tune's API** for defining search spaces.  
   - [Tune Search Space API](https://docs.ray.io/en/latest/tune/api/search_space.html)

2. **Configure the Search Algorithm**  
   - Use **Ray Tune’s Search Algorithms** for efficient exploration.  
   - [Tune Search Algorithms](https://docs.ray.io/en/latest/tune/api/suggestion.html)

3. **Implement the Objective Function**  
   - Train a **RandomForest model** using **Scikit-learn**.  
   - Optimize hyperparameters within the function.

4. **Set Up & Execute the Tuning Process**  
   - Use **Ray Tune Execution (`tune.Tuner`)** to configure and run the tuning process.  
   - Apply key configurations like:
     - `TuneConfig`
     - `RunConfig`
     - `CheckpointConfig`
     - `FailureConfig`  
   - [Tune Execution (`tune.Tuner`)](https://docs.ray.io/en/latest/tune/api/execution.html)
   - [`ray.tune.with_parameters`](https://docs.ray.io/en/latest/tune/api/doc/ray.tune.with_parameters.html)

5. **Evaluate Model Performance**  
   - Use **cross-validation** and return **negative RMSE** (to be minimized).


In [0]:
from ray import tune

# Define the hyperparameter search space for RandomForest tuning
search_space = {
    "n_estimators": tune.randint(50, 300),  # Number of trees in the forest (wider range)
    "max_depth": tune.randint(3, 30)  # Depth of the trees (realistic range)
}

In [0]:
import mlflow
from mlflow.types.utils import _infer_schema
from mlflow.exceptions import MlflowException
from mlflow.models.signature import infer_signature
from mlflow.utils.databricks_utils import get_databricks_env_vars
from ray import tune
from ray.air.integrations.mlflow import MLflowLoggerCallback, setup_mlflow
from ray.tune.search import ConcurrencyLimiter
from ray.tune.search.optuna import OptunaSearch

# Retrieve Databricks MLflow credentials
mlflow_db_creds = get_databricks_env_vars("databricks")

if not mlflow_db_creds:
    raise ValueError("Databricks MLflow credentials could not be retrieved.")

# Set up MLflow experiment
# MLflow Experiment Setup
experiment_name_ray = os.path.join(
    os.path.dirname(dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()),
    "02b - Model Tuning with Ray"
)
mlflow.set_experiment(experiment_name_ray)
experiment_id_ray = mlflow.get_experiment_by_name(experiment_name_ray).experiment_id

# Define Optuna search algorithm
searcher = OptunaSearch(metric="rmse", mode="min")  # Minimize RMSE
algo = ConcurrencyLimiter(searcher, max_concurrent=3)  # Limit concurrent trials to 3

In [0]:
import pandas as pd
import mlflow
import os
import ray
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from typing import Dict, Any

def objective_ray_scikit(
    config: Dict[str, Any],
    parent_run_id: str,
    X_train_in: pd.DataFrame,
    y_train_in: pd.Series,
    experiment_name_in: str,
    mlflow_db_creds_in: Dict[str, str]
):
    """
    Objective function for Ray Tune hyperparameter optimization using cross-validation.

    Args:
        config (Dict[str, Any]): Hyperparameter configuration from Ray Tune.
        parent_run_id (str): MLflow parent run ID for nested tracking.
        X_train_in (pd.DataFrame): Training features.
        y_train_in (pd.Series): Training labels.
        experiment_name_in (str): MLflow experiment name.
        mlflow_db_creds_in (Dict[str, str]): Databricks MLflow credentials.
    
    Returns:
        Dict[str, float]: Dictionary containing RMSE (lower is better).
    """

    try:
        # Update Databricks credentials for Ray (executors restart each run)
        if mlflow_db_creds_in:
            os.environ.update(mlflow_db_creds_in)

        # Start nested MLflow run under the parent run
        with mlflow.start_run(nested=True, experiment_id=experiment_id_ray, tags={"mlflow.parentRunId": parent_run_id}):
            # Extract hyperparameters from Ray Tune config
            n_estimators = config["n_estimators"]
            max_depth = config["max_depth"]

            # Initialize RandomForest Regressor
            model = RandomForestRegressor(
                n_estimators=n_estimators, 
                max_depth=max_depth, 
                random_state=42
            )

            # Perform 3-fold cross-validation and compute RMSE
            scores = cross_val_score(
                model, X_train_in, y_train_in, 
                scoring='neg_root_mean_squared_error', 
                cv=3
            )

            mean_rmse = -scores.mean()  # Convert negative RMSE to positive

            # Log hyperparameters and metrics in MLflow
            mlflow.log_params(config)
            mlflow.log_metric("RMSE", mean_rmse)

            return {"rmse": mean_rmse}

    except Exception as e:
        print(f"Error in objective function: {e}")
        return {"rmse": float("inf")}  # Return a large RMSE in case of failure

In [0]:
# Start the parent MLflow run
with mlflow.start_run(run_name="ray_tune", experiment_id=experiment_id_ray) as parent_run:
    os.environ.update(mlflow_db_creds)  # Ensure Ray executors have credentials

    # Set up and execute Ray Tune with the objective function and search space
    tuner = tune.Tuner(
        tune.with_parameters(
            objective_ray_scikit,
            parent_run_id=parent_run.info.run_id,
            X_train_in=X_train,
            y_train_in=y_train,
            experiment_name_in=experiment_name_ray,
            mlflow_db_creds_in=mlflow_db_creds
        ),
        tune_config=tune.TuneConfig(
            search_alg=algo,
            num_samples=10,
            reuse_actors=True  # Keeps actors alive for efficiency
        ),
        param_space=search_space
    )

    # Run tuning and retrieve the best result
    multinode_results = tuner.fit()
    best_result = multinode_results.get_best_result(metric="rmse", mode="min", scope="last")

    if best_result is None:
        raise ValueError("No best trial found. Ensure the tuning job ran successfully.")

    # Extract best trial details
    best_trial_number = best_result.metrics.get("trial_id", "N/A")  # Default if missing
    best_model_params = best_result.config
    best_model_params["random_state"] = 42  # Ensures reproducibility
    best_rmse = best_result.metrics["rmse"]

    # Train the best model using the best hyperparameters
    best_model = RandomForestRegressor(**best_model_params)
    best_model.fit(X_train, y_train)

    # Enable MLflow autologging (disable model logging to avoid conflicts)
    mlflow.sklearn.autolog(log_input_examples=True, log_models=False, silent=True)

    # Infer model output schema
    try:
        output_schema = _infer_schema(y_train)
    except Exception as e:
        warnings.warn(f"Could not infer model output schema: {e}")
        output_schema = None

    # Infer model signature
    input_example = X_train[:3]  # Use a small subset as an example
    signature = infer_signature(X_train, best_model.predict(X_train))

    # Set model name for MLflow registration
    model_name = f"{DA.catalog_name}.{DA.schema_name}.hpo_model_ray_tune_optuna"

    # Display results
    print(f"Best Trial Number: {best_trial_number}")
    print(f"Best Hyperparameters: {best_model_params}")
    print(f"Best RMSE: {best_rmse:.4f}")

    # Log the best model to MLflow
    with mlflow.start_run(run_name="best_trial_ray_scikit_results", nested=True):
        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="model",
            signature=signature,
            input_example=input_example,
            registered_model_name=model_name
        )
        mlflow.log_params(best_model_params)
        mlflow.log_metric("Best RMSE", best_rmse)

# Ensure MLflow run is properly closed
mlflow.end_run()

2025-09-23 19:57:48,806	INFO worker.py:1630 -- Using address 10.3.33.157:9813 set in the environment variable RAY_ADDRESS
2025-09-23 19:57:48,808	INFO worker.py:1771 -- Connecting to existing Ray cluster at address: 10.3.33.157:9813...
2025-09-23 19:57:48,817	INFO worker.py:1951 -- Connected to Ray cluster.
2025-09-23 19:57:48,835	INFO tune.py:253 -- Initializing Ray automatically. For cluster usage or custom Ray initialization, call `ray.init(...)` before `Tuner(...)`.
[I 2025-09-23 19:57:48,845] A new study created in memory with name: optuna


+-----------------------------------------------------------------------------+
| Configuration for experiment     objective_ray_scikit_2025-09-23_19-57-48   |
+-----------------------------------------------------------------------------+
| Search algorithm                 SearchGenerator                            |
| Scheduler                        FIFOScheduler                              |
| Number of trials                 10                                         |
+-----------------------------------------------------------------------------+

View detailed results here: /root/ray_results/objective_ray_scikit_2025-09-23_19-57-48
To visualize your results with TensorBoard, run: `tensorboard --logdir /local_disk0/tmp/ray-9813-acf1b11b/session_2025-09-23_19-43-01_327811_17064/artifacts/2025-09-23_19-57-48/objective_ray_scikit_2025-09-23_19-57-48/driver_artifacts`

Trial status: 1 PENDING
Current time: 2025-09-23 19:57:52. Total running time: 0s
Logical resource usage: 0/4 CPUs,

(objective_ray_scikit pid=25487) Tue Sep 23 19:57:57 2025 Connection to spark from PID  25487
(objective_ray_scikit pid=25487) Tue Sep 23 19:57:57 2025 Initialized gateway on port 41671
(objective_ray_scikit pid=25487) Tue Sep 23 19:57:57 2025 Connection to spark from PID  25487
(objective_ray_scikit pid=25487) Tue Sep 23 19:57:57 2025 Initialized gateway on port 41671
(objective_ray_scikit pid=25487) Tue Sep 23 19:57:57 2025 Connected to spark.
(objective_ray_scikit pid=25487) Tue Sep 23 19:57:57 2025 Connected to spark.



Trial objective_ray_scikit_6f6f980f completed after 1 iterations at 2025-09-23 19:57:59. Total running time: 7s
+--------------------------------------------------------+
| Trial objective_ray_scikit_6f6f980f result             |
+--------------------------------------------------------+
| checkpoint_dir_name                                    |
| time_this_iter_s                                3.5928 |
| time_total_s                                    3.5928 |
| training_iteration                                   1 |
| rmse                                           0.66069 |
+--------------------------------------------------------+

Trial objective_ray_scikit_bd2b7191 started with configuration:
+----------------------------------------------------+
| Trial objective_ray_scikit_bd2b7191 config         |
+----------------------------------------------------+
| max_depth                                        6 |
| n_estimators                                   255 |
+---------------

2025-09-23 19:58:22,595	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/root/ray_results/objective_ray_scikit_2025-09-23_19-57-48' in 0.0158s.


...

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8ca94bc5-3683-4814-9378-dffc69f99992/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Best Trial Number: 61405d3f
Best Hyperparameters: {'n_estimators': 67, 'max_depth': 23, 'random_state': 42}
Best RMSE: 0.6572


2025/09/23 19:58:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-acab7dc4-7199.cloud.databricks.com/ml/experiments/1903700905084210/models/m-0a296c43b55d496b8ad446a4d42d9cc4?o=3037386709546306
Successfully registered model 'dbacademy.labuser11731658_1758653716.hpo_model_ray_tune_optuna'.


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '1' of model 'dbacademy.labuser11731658_1758653716.hpo_model_ray_tune_optuna': https://dbc-acab7dc4-7199.cloud.databricks.com/explore/data/models/dbacademy/labuser11731658_1758653716/hpo_model_ray_tune_optuna/version/1?o=3037386709546306


In [0]:
print(best_result, "\n")
print(best_result.metrics, "\n")
print(best_result.config, "\n")
print(best_model_params)

Result(
  metrics={'rmse': 0.6571720129864256},
  path='/root/ray_results/objective_ray_scikit_2025-09-23_19-57-48/objective_ray_scikit_61405d3f_5_max_depth=23,n_estimators=67_2025-09-23_19-58-06',
  filesystem='local',
  checkpoint=None
) 

{'rmse': 0.6571720129864256, 'timestamp': 1758657490, 'checkpoint_dir_name': None, 'done': True, 'training_iteration': 1, 'trial_id': '61405d3f', 'date': '2025-09-23_19-58-10', 'time_this_iter_s': 1.4143123626708984, 'time_total_s': 1.4143123626708984, 'pid': 25487, 'hostname': '0923-185603-ggr7be56-10-3-33-157', 'node_ip': '10.3.33.157', 'config': {'n_estimators': 67, 'max_depth': 23, 'random_state': 42}, 'time_since_restore': 1.4143123626708984, 'iterations_since_restore': 1, 'experiment_tag': '5_max_depth=23,n_estimators=67'} 

{'n_estimators': 67, 'max_depth': 23, 'random_state': 42} 

{'n_estimators': 67, 'max_depth': 23, 'random_state': 42}


Shut down the ray cluster.

In [0]:
ray.shutdown()

# Conclusion

In this demo, we explored how to setup and execute model training with Ray Tune on a single-machine. Additionally, we walked through the core components needed to perform model training with the Ray framework such as defining a search space, building objective functions, and optimization of hyperparameters.


&copy; 2025 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="blank">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy" target="blank">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use" target="blank">Terms of Use</a> | 
<a href="https://help.databricks.com/" target="blank">Support</a>